# QEC code search on Colab

This notebook runs an evolutionary search for quantum error-correcting codes in the family behind IBM's gross code. It saves the results to your Google Drive, and it writes a short report for you to send back to Claude.

**Before you start:** use a **CPU** runtime. The search doesn't use a GPU. Go to *Runtime → Change runtime type → CPU*.

**How to use it:** run the cells from top to bottom. The search cell runs for as long as you set. Keep this tab open, because Colab disconnects idle sessions. If Colab disconnects, run cells 1 and 3 again with the same run name, and the search picks up where it stopped.

In [ ]:
#@title 1 · Setup (clone our fork, install the lab package + Stim + ldpc, connect Drive)
BRANCH = "main"         #@param {type:"string"}
SAVE_TO_DRIVE = True    #@param {type:"boolean"}

import os, subprocess, sys

REPO = "https://github.com/fullymiddleaged/qldpc-challenge-foundry"
ROOT = "/content/qldpc-challenge-foundry"
LAB = f"{ROOT}/lab"
if not os.path.exists(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO, ROOT], check=True)
else:  # re-running the cell picks up new commits
    subprocess.run(["git", "-C", ROOT, "pull", "-q", "--ff-only"], check=True)
subprocess.run(["git", "-C", ROOT, "log", "-1", "--format=lab code at %h  %s"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{LAB}[sim]"], check=True)
os.chdir(LAB)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS_DIR = "/content/drive/MyDrive/qec-runs"
else:
    RUNS_DIR = "/content/runs"   # wiped when the runtime ends
os.makedirs(RUNS_DIR, exist_ok=True)

import stim, ldpc
print(f"CPU cores: {os.cpu_count()} | stim {stim.__version__} | ldpc {getattr(ldpc, '__version__', '?')}")
print(f"results will be saved in: {RUNS_DIR}")


In [ ]:
#@title 2 · Self-test (about 1–3 minutes). It must end with ALL CHECKS PASSED
!cd {LAB} && python -u -m qec_search.selftest

### Choosing a preset
- **ibm-style**: weight-6 checks built from pure powers of x and y, the same shape as IBM's codes. Most of these small search spaces can be covered in a few hours, and they have probably been enumerated in published work already. Run it once as a baseline and a sanity check.
- **mixed**: allows terms like x²y³. The search space is millions of times larger and much less explored. **This is where a new result is most likely.**
- **weight-7**: one extra term in B, giving checks that touch 7 qubits. Hardware wiring gets harder, and distance can go up.

In [ ]:
#@title 3 · Run the search
RUN_NAME = "run1"      #@param {type:"string"}
PRESET = "mixed"       #@param ["ibm-style", "mixed", "weight-7"]
HOURS = 2.0            #@param {type:"number"}
EXTRA_FLAGS = ""       #@param {type:"string"}

import os
OUT = f"{RUNS_DIR}/{RUN_NAME}"
flags = {"ibm-style": "--seed-known",
         "mixed": "--mixed --seed-known",
         "weight-7": "--terms 3,4"}[PRESET]
cmd = (f"python -u -m qec_search.search --out '{OUT}' --hours {HOURS} "
       f"--workers {os.cpu_count()} {flags} {EXTRA_FLAGS}")
print(cmd, "\n")
!cd {LAB} && {cmd}

In [ ]:
#@title 4 · Summarise and show the report
!cd {LAB} && python -m qec_search.summarize "{OUT}" > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/report_for_claude.md").read()))

### 5 · Optional: circuit-level check of the best finds

This step simulates the full noisy measurement circuit, which is slower but much more realistic. It runs the top candidates next to the IBM code of the same size as a control. The settings here (4 rounds, 2,000 shots) are sized for Colab. The cell prints the time for each code, so you can see what bigger settings would cost before trying them.

In [ ]:
#@title 5 · Circuit-level verification
TOP = 3          #@param {type:"integer"}
NOISE = "0.003"  #@param {type:"string"}
ROUNDS = 4       #@param {type:"integer"}
SHOTS = 2000     #@param {type:"integer"}
!cd {LAB} && python -u -m qec_search.verify "{OUT}" --top {TOP} --ps {NOISE} --rounds {ROUNDS} --shots {SHOTS}
!cd {LAB} && python -m qec_search.summarize "{OUT}" > /dev/null
print(f"report updated: {OUT}/report_for_claude.md")

### 6 · Dry-run the record candidates Claude prepared

These are five codes that Claude has already given a **2D bilayer layout**, so they fall into the less crowded *2D-local bilayer, weight ≤ 6* cell: four codes from your baseline run, plus IBM's gross code. Claude's own checks put each one within every rule, with a largest check diameter of 7.0 or less and at most 2 qubits per site. This cell runs the challenge's own checker on them in **dry-run** mode, so nothing is submitted.

The gross code is IBM's, not ours. The layout would be our contribution, so if you ever submit it, credit IBM, as its construction note already does.

In [ ]:
#@title 6 · Dry-run the record candidates (nothing is submitted)
GITHUB_HANDLE = "@yourhandle"   #@param {type:"string"}
MODEL = "claude-opus-5-5"       #@param {type:"string"}
import json, os
DIR = f"{LAB}/results/candidates"
if not os.path.exists("/content/qldpc-challenge"):
    !git clone -q --depth 1 https://github.com/unitaryfoundation/qldpc-challenge /content/qldpc-challenge
!pip install -q uv
LOG = f"{RUNS_DIR}/checker_log_candidates.txt"
!cd /content/qldpc-challenge && (./qldpc targets --n 300 --top 10 || ./qldpc targets --help) 2>&1 | tee "{LOG}"
manifest = json.load(open(f"{DIR}/manifest.json"))
model_flag = f'--model "{MODEL}"' if MODEL else ""
for name, info in manifest.items():
    print("\n" + "=" * 90 + f"\n{name}  {info['params']}  layout radius {info['radius']}\n" + "=" * 90)
    f = f"{DIR}/{name}.npz"
    constr = info["construction"]
    !cd /content/qldpc-challenge && ./qldpc submit "{f}" --authors "{GITHUB_HANDLE}" --family bivariate-bicycle --construction "{constr}" {model_flag} --layers 2 --coords "{f}" --no-circuit --dry-run 2>&1 | tee -a "{LOG}"
print(f"\nsaved: {LOG}  (send this to Claude)")

### 6b · For later runs: export, lay out and dry-run the best codes of any run

This does the same as cell 6 for any run folder. It picks the run's best distinct codes, searches for a bilayer layout for each one (1–4 minutes per code), and then dry-runs the checker on them.

In [ ]:
#@title 6b · Export + layout + dry-run from a run
RUN_NAME = "mixed1"             #@param {type:"string"}
GITHUB_HANDLE = "@yourhandle"   #@param {type:"string"}
MODEL = "claude-opus-5-5"       #@param {type:"string"}
N_CODES = 6                     #@param {type:"integer"}
import json, os
OUT = f"{RUNS_DIR}/{RUN_NAME}"
if not os.path.exists("/content/qldpc-challenge"):
    !git clone -q --depth 1 https://github.com/unitaryfoundation/qldpc-challenge /content/qldpc-challenge
!pip install -q uv
!rm -rf /content/submissions
!cd {LAB} && python -u -m qec_search.export "{OUT}" --top {N_CODES} --dest /content/submissions --layout
LOG = f"{OUT}/checker_log.txt"
manifest = json.load(open("/content/submissions/manifest.json"))
model_flag = f'--model "{MODEL}"' if MODEL else ""
!echo "" > "{LOG}"
for name, info in manifest.items():
    print("\n" + "=" * 90 + f"\n{name}  {info['params']}  layers {info['layers']}\n" + "=" * 90)
    f = f"/content/submissions/{name}.npz"
    constr = info["construction"]
    lay = f'--layers 2 --coords "{f}"' if info.get("layers") == 2 else ""
    !cd /content/qldpc-challenge && ./qldpc submit "{f}" --authors "{GITHUB_HANDLE}" --family bivariate-bicycle --construction "{constr}" {model_flag} {lay} --no-circuit --dry-run 2>&1 | tee -a "{LOG}"
print(f"\nsaved: {LOG}  (send this to Claude)")

### 7 · Prove a code's exact distance (long job)

This proves the true minimum distance with an integer-programming solver, rather than just finding an upper bound. For our 216- and 288-qubit codes with distance 16 it took about 25–30 minutes on Claude's machine, which is already done. **[[288,8,20]] needs several hours**, so run it overnight with this tab open. If it hits the time limit, it still reports a *proven* minimum ("distance ≥ X"). Only one cell can run at a time, so don't run the search while this is going.

In [ ]:
#@title 7 · Prove exact distance (leave running; several hours for [[288,8,20]])
CANDIDATE = "[[288,8,20]]"  #@param ["[[288,8,20]]", "[[288,12,16]]", "[[216,8,16]]"]
HOURS = 10                  #@param {type:"number"}
G = {"[[288,8,20]]":  ('{"l":24,"m":6,"A":[[0,1],[3,0],[4,0]],"B":[[0,3],[1,0],[8,0]]}', 20),
     "[[288,12,16]]": ('{"l":24,"m":6,"A":[[0,3],[1,0],[14,0]],"B":[[0,1],[0,2],[3,0]]}', 16),
     "[[216,8,16]]":  ('{"l":18,"m":6,"A":[[0,5],[1,0],[12,0]],"B":[[0,1],[2,0],[3,0]]}', 16)}
genome, claimed = G[CANDIDATE]
tag = CANDIDATE.strip("[]").replace(",", "_")
PROOF = f"{RUNS_DIR}/distance_proof_{tag}.json"
!cd {LAB} && python -u -m qec_search.exact_distance --genome '{genome}' --claimed {claimed} --hours {HOURS} --out "{PROOF}" 2>&1 | grep -v transformNewInteger
print(f"result saved to {PROOF} (send this file to Claude)")

In [ ]:
#@title 8 · Download the files to send to Claude
import os
from google.colab import files
OUT = globals().get("OUT", f"{RUNS_DIR}/baseline")
import glob
for path in [f"{OUT}/report_for_claude.md", f"{OUT}/checker_log.txt", f"{RUNS_DIR}/checker_log_candidates.txt"] + glob.glob(f"{RUNS_DIR}/distance_proof_*.json"):
    if os.path.exists(path):
        files.download(path)